# Оригинальная VMD-TiDE-12: обучение и прогноз F10.7 на 7 дней

Откройте **Среда выполнения → Сменить среду выполнения → GPU** (CPU также поддерживается), задайте каталог ниже и запускайте ячейки сверху вниз. Подключение Drive требует стандартного разрешения Google. При необходимости установки ячейка попросит один раз перезапустить сеанс и повторить запуск сверху.

Используются авторские `ForecastModel`, `TideModel`, двухэтапный VMD и `hyperparams/default.env`. Никаких дополнительных признаков, EarlyStopping или других моделей. Обучение здесь не подтверждает воспроизведение метрик статьи. Полный VMD всей истории может потребовать нескольких ГБ RAM; история не обрезается. Декомпозиция всей истории соответствует авторскому коду и не является причинной схемой для исторического бэктеста.


## 1. Configuration

Веса никогда не перезаписываются: отдельный запуск для каждой попытки компоненты. Возобновляются **завершённые компоненты**, незавершённая компонента обучается заново в новом каталоге. Для обычного прогноза установите `TRAIN_MISSING = False`.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, contextlib, uuid, platform
from datetime import datetime, timezone

RESULTS_DIR = Path("/content/drive/MyDrive/ИСЗФ/VMD_TiDE/orig")
MOUNT_DRIVE = True
TRAIN_MISSING = True
REPO_URL = "https://github.com/Yuri-ga1/VMD-TiDE.git"
REPO_BRANCH = "original-model"
REPO_DIR = Path("/content/VMD-TiDE")
HORIZON, CONTEXT = 7, 126
TRAIN_START, TRAIN_END = "1954-01-01", "1986-08-31"
VAL_START, VAL_END = "1986-09-01", "2008-12-31"
TEST_START, TEST_END = "2009-01-01", None
# split_after включает граничную дату в train.
SPLIT_AFTER = TRAIN_END
MAX_EPOCHS, BATCH_SIZE, SEED = 50, 512, 42
VMD_PARAMS = dict(K=3, alpha=2626, tau=0, DC=0, init=1, tol=1e-7)

print("Python:", sys.version)
# Отдельный процесс не загружает torch в ядро до установки зависимостей.
subprocess.run([sys.executable, "-c", """
try:
    import torch
    print("PyTorch до установки:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    print("Устройство:", "cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    print("PyTorch ещё не установлен; устройство будет определено после установки.")
"""], check=True)


## 2. Clone/open repository

In [ ]:
def git_output(path, *args):
    return subprocess.check_output(["git", "-C", str(path), *args], text=True).strip()

candidates = [REPO_DIR, Path.cwd(), Path("/content/VMD-TiDE")]
REPO = None
for candidate in candidates:
    if (candidate / ".git").exists():
        branch = git_output(candidate, "branch", "--show-current")
        remote = git_output(candidate, "remote", "get-url", "origin").removesuffix(".git")
        if branch == REPO_BRANCH and remote == REPO_URL.removesuffix(".git"):
            REPO = candidate.resolve()
            break
if REPO is None:
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} уже существует и не подходит. Задайте другой REPO_DIR; каталог сохранён.")
    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"Не удалось получить {REPO_BRANCH}. Проверьте наличие ветки и доступ GitHub.\n" + result.stderr)
    REPO = REPO_DIR.resolve()
required = ["requirements.txt", "migrate.py", "data.sql", "solar_cycles.sql",
            "classes/ForecastModel.py", "classes/TideModel.py", "hyperparams/default.env"]
for name in required:
    if not (REPO / name).is_file():
        raise FileNotFoundError(name)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
SOURCE_COMMIT = git_output(REPO, "rev-parse", "HEAD")
print("Рабочая копия:", REPO, "\nВетка:", REPO_BRANCH, "\nCommit:", SOURCE_COMMIT)


## 3. Mount Google Drive

Для сохранения между сессиями оставьте `MOUNT_DRIVE = True`. При отключении Drive задайте локальный `RESULTS_DIR`; такая копия пропадёт при удалении runtime. Секреты PostgreSQL на Drive не сохраняются.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
elif str(RESULTS_DIR).startswith("/content/drive/"):
    raise RuntimeError("Каталог на Drive требует MOUNT_DRIVE = True.")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Результаты:", RESULTS_DIR)

def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json_new(path, value):
    # Режим x не допускает перезаписи существующего файла.
    with open(path, "x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)

@contextlib.contextmanager
def working_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


## 4. Install dependencies

In [ ]:
import importlib.metadata as metadata
import re, shutil, tempfile

if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError("Этот профиль рассчитан на Python 3.10–3.13. Для иной версии нужны совместимые wheels.")

PINS = {
    "requests": "requests==2.32.5", "sqlalchemy": "SQLAlchemy==2.0.43",
    "darts": "u8darts[torch]==0.38.0", "joblib": "joblib==1.5.2",
    "lightning": "lightning==2.5.2", "matplotlib": "matplotlib==3.10.6",
    "matplotlib-venn": "matplotlib-venn==1.1.2", "numpy": "numpy==2.2.6",
    "pandas": "pandas==2.2.3", "psycopg2-binary": "psycopg2-binary==2.9.10",
    "python-dotenv": "python-dotenv==1.1.1", "python-crontab": "python-crontab==3.3.0",
    "scikit-learn": "scikit-learn==1.6.1", "scipy": "scipy==1.15.3",
    "stdlib-list": "stdlib-list==0.11.1", "sunpy": "sunpy==6.1.2",
    "torch": "torch==2.8.0", "vmdpy": "vmdpy==0.2",
}
def normalize(name):
    return re.sub(r"[-_.]+", "-", name).lower()

original_requirements = (REPO / "requirements.txt").read_text().splitlines()
install_specs = []
for line in original_requirements:
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    name = normalize(re.split(r"[<>=!~\[; ]", line)[0])
    if name not in PINS:
        raise RuntimeError(f"Новая зависимость автора {line}: профиль требует пересмотра.")
    install_specs.append(PINS[name])
install_specs += ["pytorch-lightning==2.5.2", "torchmetrics==1.8.2",
                  "torchvision==0.23.0", "torchaudio==2.8.0", "librosa==0.11.0"]
# torchmetrics может импортировать librosa: исключаем старый код с numpy.complex.
loaded_before = {}
for module_name, module in list(sys.modules.items()):
    if "." not in module_name and getattr(module, "__version__", None):
        loaded_before[module_name] = str(module.__version__)

nvidia = shutil.which("nvidia-smi")
has_nvidia = bool(nvidia and subprocess.run([nvidia], capture_output=True).returncode == 0)
index = "https://download.pytorch.org/whl/cu126" if has_nvidia else "https://download.pytorch.org/whl/cpu"
torch_specs = ["torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"]
# Весь профиль разрешается вместе, до изменения окружения.
pip_base = [sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed"]
subprocess.run(pip_base + ["--dry-run", "--extra-index-url", index] + install_specs, check=True)
subprocess.run(pip_base + ["--index-url", index] + torch_specs, check=True)
subprocess.run(pip_base + install_specs, check=True)

changed = []
for mod, version in loaded_before.items():
    dist = {"sklearn": "scikit-learn", "PIL": "pillow"}.get(mod, mod)
    try:
        if metadata.version(dist).split("+")[0] != version.split("+")[0]:
            changed.append(mod)
    except metadata.PackageNotFoundError:
        pass
if changed:
    raise RuntimeError(
        "Обновлены загруженные библиотеки: " + ", ".join(changed) +
        ". Выберите Среда выполнения → Перезапустить сеанс, затем Выполнить все. "
        "Не выбирайте удаление среды. Установленные версии уже сохранены.")

import torch
from darts.models import TiDEModel
import pytorch_lightning, numpy as np, pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ACCELERATOR = "gpu" if DEVICE == "cuda" else "cpu"

print("Python:", platform.python_version(), "| PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available(), "| устройство:", DEVICE)
print("Darts:", metadata.version("u8darts"), "| Lightning:", metadata.version("pytorch-lightning"))


## 5. Initialize PostgreSQL

In [ ]:
import secrets
from dotenv import dotenv_values
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "postgresql", "postgresql-contrib"], check=True)
subprocess.run(["service", "postgresql", "start"], check=True)

def admin_sql(statement):
    # SQL с временным паролем передаётся через stdin и не печатается.
    result = subprocess.run(
        ["runuser", "-u", "postgres", "--", "psql", "-X", "-v", "ON_ERROR_STOP=1", "-d", "postgres"],
        input=statement, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError("Не удалось настроить локальный PostgreSQL; секретный SQL скрыт.")

env_path = REPO / ".env"
local_names = dict(DB_USER="vmd_colab", DB_NAME="vmd_colab", DB_HOST="127.0.0.1",
                   DB_PORT="5432", DB_SCHEMA="f107")
if env_path.exists():
    config = dict(dotenv_values(env_path))
    if any(config.get(k) != v for k, v in local_names.items()):
        raise RuntimeError("Существующий .env не является конфигурацией этой тетради. Выберите отдельный REPO_DIR.")
else:
    config = {**local_names, "DB_PASSWORD": secrets.token_hex(24)}
    with open(env_path, "x", encoding="utf-8") as stream:
        stream.write("\n".join(f"{k}={v}" for k, v in config.items()) + "\n")
    env_path.chmod(0o600)
password = config["DB_PASSWORD"]
if not re.fullmatch(r"[a-f0-9]{48}", password):
    raise RuntimeError("Пароль локальной роли не соответствует формату этой тетради.")
admin_sql(fr"""
DO $$ BEGIN
IF NOT EXISTS (SELECT FROM pg_roles WHERE rolname='vmd_colab') THEN
    CREATE ROLE vmd_colab LOGIN;
END IF;
END $$;
ALTER ROLE vmd_colab PASSWORD '{password}';
SELECT 'CREATE DATABASE vmd_colab OWNER vmd_colab'
WHERE NOT EXISTS (SELECT FROM pg_database WHERE datname='vmd_colab')\gexec
""")
os.environ.update(config)
engine = create_engine(URL.create("postgresql+psycopg2", username=config["DB_USER"],
    password=password, host=config["DB_HOST"], port=int(config["DB_PORT"]), database=config["DB_NAME"]))
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS f107 AUTHORIZATION vmd_colab"))

db_path = REPO / "functions/db.py"
source = db_path.read_text()
old_path = 'dotenv_values("/home/user/projects/f107/.env")'
if old_path in source:
    source = source.replace("import os\n", "import os\nfrom pathlib import Path\n", 1)
    helper = """
def _db_config():
    project_env = Path(__file__).resolve().parents[1] / '.env'
    return dotenv_values(project_env if project_env.is_file() else '/home/user/projects/f107/.env')

"""
    if "def _db_config():" not in source:
        source += helper
    source = source.replace(old_path, "_db_config()")
    db_path.write_text(source)
elif "def _db_config():" not in source:
    raise RuntimeError("Неизвестная версия functions/db.py.")

migration = REPO / "migrate.py"
source = migration.read_text()
old = "Base.metadata.create_all(engine, tables=[Base.metadata.tables[table] for table in tables_to_create])"
new = "Base.metadata.create_all(engine, tables=[table for table in Base.metadata.sorted_tables if table.name in tables_to_create])"
if old in source:
    migration.write_text(source.replace(old, new))
elif "Base.metadata.sorted_tables" not in source:
    raise RuntimeError("Неизвестная версия migrate.py.")
import importlib
importlib.invalidate_caches()
import migrate
migrate.setup_database(schema="f107")
print("Локальная БД и таблицы подготовлены.")


## 6. Restore historical data

In [ ]:
# Дампы автора содержат INSERT; данные файлов остаются неизменными.
import ast
def dump_values(path, table):
    pattern = re.compile(r'INSERT INTO "f107"\."' + table + r'" VALUES (\(.*\));')
    values = []
    for line in path.read_text().splitlines():
        match = pattern.fullmatch(line.strip())
        if match:
            values.append(ast.literal_eval(match.group(1)))
        elif line.strip() and not line.lstrip().startswith("--"):
            raise RuntimeError(f"Неизвестный формат дампа: {path.name}")
    if not values:
        raise RuntimeError(f"Пустой дамп: {path.name}")
    return values

for table, filename in [("data", "data.sql"), ("solar_cycles", "solar_cycles.sql")]:
    path = REPO / filename
    expected = dump_values(path, table)
    with engine.begin() as connection:
        count = connection.execute(text(f"SELECT count(*) FROM f107.{table}")).scalar_one()
        if count == 0:
            # Оригинальная функция подавляет ошибки; исполняем тот же SQL в транзакции
            # непосредственно драйвером, чтобы при сбое импорт целиком откатывался.
            with connection.connection.cursor() as cursor:
                cursor.execute(path.read_text())
        rows = connection.execute(text(f"SELECT * FROM f107.{table} ORDER BY id")).fetchall()
        def canonical(row):
            return [v.isoformat(sep=" ") if isinstance(v, datetime)
                    else v.isoformat() if hasattr(v, "isoformat") else v for v in row]
        actual = [canonical(row) for row in rows]
        if actual != [list(row) for row in expected]:
            raise RuntimeError(f"{table}: содержимое отличается от {filename}; ничего не удалено.")
        connection.execute(text(
            f"SELECT setval(pg_get_serial_sequence('f107.{table}', 'id'), "
            f"(SELECT max(id) FROM f107.{table}), true)"))
    print(table, ":", len(rows), "строк; импорт подтверждён.")

import functions.db as db
from classes.ForecastModel import ForecastModel
from classes.TideModel import TideModel
observed = db.select_rows("SELECT * FROM data ORDER BY date")
observed["date"] = pd.to_datetime(observed["date"])
if observed["date"].duplicated().any() or observed["real"].isna().any():
    raise RuntimeError("В наблюдениях есть повторные даты или пропуски F10.7.")
if not np.isfinite(observed["real"]).all():
    raise RuntimeError("В наблюдениях есть нечисловые значения.")
if not pd.DatetimeIndex(observed["date"]).equals(
        pd.date_range(observed["date"].min(), observed["date"].max(), freq="D")):
    raise RuntimeError("Наблюдения должны составлять непрерывный ежедневный ряд.")
LAST_DATE = observed["date"].max()
if observed["date"].min() > pd.Timestamp(TRAIN_START) or LAST_DATE < pd.Timestamp(VAL_END):
    raise RuntimeError("История не покрывает заданные интервалы обучения и валидации.")
print("Наблюдения:", observed["date"].min().date(), "—", LAST_DATE.date())
if LAST_DATE.date() < datetime.now(timezone.utc).date():
    print("Данные заканчиваются в прошлом. Прогноз будет после", LAST_DATE.date(),
          ", а не после сегодняшней даты.")


## 7. Prepare VMD decomposition

In [ ]:
import joblib
import inspect
if {k: inspect.signature(ForecastModel.vmd_decompose).parameters[k].default
        for k in VMD_PARAMS} != VMD_PARAMS:
    raise RuntimeError("Параметры VMD исходника отличаются от ожидаемых.")
COMPONENTS = ForecastModel(HORIZON).modes
if len(COMPONENTS) != 12 or len(set(COMPONENTS)) != 12:
    raise RuntimeError("Ожидались 12 авторских компонент.")
DECOMP_CONFIG = {
    "observations": signature(observed.to_json(date_format="iso", double_precision=15)),
    "vmd": VMD_PARAMS, "vmdpy": metadata.version("vmdpy"),
    "source": sha_file(REPO / "classes/ForecastModel.py"),
}
cache_dir = RESULTS_DIR / "decomposition" / signature(DECOMP_CONFIG)
cache_dir.mkdir(parents=True, exist_ok=True)
decomposed = None
for manifest_path in sorted(cache_dir.glob("*.json")):
    manifest = json.loads(manifest_path.read_text())
    payload = cache_dir / manifest["file"]
    if manifest["config"] == DECOMP_CONFIG and payload.is_file() and sha_file(payload) == manifest["sha256"]:
        decomposed = joblib.load(payload)
        print("Загружен совместимый кэш VMD:", payload.name)
        break
if decomposed is None:
    print("Полный двухэтапный VMD всей истории; это может занять время и несколько ГБ RAM.")
    decomposition_model = ForecastModel(HORIZON, data_range=[TRAIN_START, LAST_DATE.strftime("%Y-%m-%d")])
    decomposition_model.prepare_data_decomposed()
    decomposed = db.select_rows("SELECT * FROM data_decomposed ORDER BY date")
    token = uuid.uuid4().hex
    payload = cache_dir / f"{token}.joblib"
    joblib.dump(decomposed, payload)
    write_json_new(cache_dir / f"{token}.json",
                   {"config": DECOMP_CONFIG, "file": payload.name, "sha256": sha_file(payload)})
decomposed["date"] = pd.to_datetime(decomposed["date"])
if decomposed["date"].duplicated().any() or decomposed["date"].max() != LAST_DATE:
    raise RuntimeError("Даты VMD не соответствуют последним наблюдениям.")
if not np.isfinite(decomposed[COMPONENTS].to_numpy()).all():
    raise RuntimeError("VMD содержит пропуски или бесконечные значения.")
if not pd.DatetimeIndex(decomposed["date"]).equals(
        pd.date_range(decomposed["date"].min(), LAST_DATE, freq="D")):
    raise RuntimeError("После VMD нарушена ежедневная частота.")

db.insert_df("data_decomposed", decomposed.set_index("id"))
print("data_decomposed:", len(decomposed), "строк. Порядок:", COMPONENTS)


## 8. Train VMD-TiDE-12

In [ ]:
import importlib.metadata as metadata
import shutil, tempfile, inspect

if (HORIZON, CONTEXT, MAX_EPOCHS, BATCH_SIZE, SEED) != (7, 126, 50, 512, 42):
    raise RuntimeError("This notebook requires the original 7/126/50/512/42 settings.")
VERSIONS = {name: metadata.version(name) for name in
    ["torch", "u8darts", "lightning", "pytorch-lightning", "numpy", "pandas", "scikit-learn", "scipy", "joblib", "vmdpy"]}
VERSIONS["torch"] = VERSIONS["torch"].split("+")[0]
MODEL_DIR = RESULTS_DIR / "models"
SCALER_DIR = RESULTS_DIR / "scalers"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
FORECAST_DIR = RESULTS_DIR / "forecasts"
for folder in (MODEL_DIR, SCALER_DIR, CHECKPOINT_DIR, FORECAST_DIR):
    folder.mkdir(parents=True, exist_ok=True)
LOCAL_TRAIN_DIR = Path("/content/vmd_tide_training")
DARTS_WORK_DIR = LOCAL_TRAIN_DIR / "darts_logs"
LOCAL_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
DARTS_WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CONFIG = {
    "vmd": VMD_PARAMS,
    "tide": {"context": CONTEXT, "horizon": HORIZON, "epochs": MAX_EPOCHS,
             "batch_size": BATCH_SIZE, "seed": SEED,
             "hyperparameters": (REPO / "hyperparams/default.env").read_text()},
    "train": [TRAIN_START, TRAIN_END], "validation": [VAL_START, VAL_END],
    "split_after": SPLIT_AFTER, "components": COMPONENTS,
    "repository_commit": SOURCE_COMMIT,
    "data_sha256": signature(observed.to_json(date_format="iso", double_precision=15)),
    "versions": VERSIONS,
    "source_sha256": {name: sha_file(REPO / name) for name in
        ["classes/ForecastModel.py", "classes/TideModel.py", "functions/utilites.py"]},
}
CONFIG_PATH = RESULTS_DIR / "config.json"
CONFIG_HASH = signature(TRAIN_CONFIG)
if CONFIG_PATH.exists():
    config_state = json.loads(CONFIG_PATH.read_text())
    if config_state.get("configuration") != TRAIN_CONFIG:
        completed = config_state.get("completed_components", {})
        saved_models = any(MODEL_DIR.glob("*.pt")) or any(SCALER_DIR.glob("*.joblib"))
        saved_checkpoints = any(CHECKPOINT_DIR.rglob("*.ckpt")) or any(CHECKPOINT_DIR.rglob("_model.pth.tar"))
        if completed or saved_models or saved_checkpoints:
            differences = [key for key in set(config_state.get("configuration", {})) | set(TRAIN_CONFIG)
                           if config_state.get("configuration", {}).get(key) != TRAIN_CONFIG.get(key)]
            raise RuntimeError(
                "Drive config.json differs in " + ", ".join(differences) +
                ". Registered models/checkpoints were preserved and were not loaded. "
                "Use their matching notebook/configuration or set RESULTS_DIR to a new folder.")
        # Only stale metadata remains. Keep it as an audit copy and start the requested fresh run.
        backup = RESULTS_DIR / ("config.previous." + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + ".json")
        shutil.copy2(CONFIG_PATH, backup)
        config_state = {"configuration": TRAIN_CONFIG, "completed_components": {}}
        print("No model or checkpoint artifacts were found. Previous config preserved at:", backup.name)
else:
    config_state = {"configuration": TRAIN_CONFIG, "completed_components": {}}

def atomic_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as stream:
            json.dump(payload, stream, ensure_ascii=False, indent=2)
            stream.flush(); os.fsync(stream.fileno())
        os.replace(tmp, path)
    except Exception:
        try: os.unlink(tmp)
        except FileNotFoundError: pass
        raise

def atomic_copy(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=destination.parent, prefix=destination.name + ".", suffix=".tmp")
    os.close(fd)
    try:
        shutil.copy2(source, tmp)
        if sha_file(source) != sha_file(tmp):
            raise IOError(f"Copy verification failed: {destination.name}")
        os.replace(tmp, destination)
    except Exception:
        try: os.unlink(tmp)
        except FileNotFoundError: pass
        raise

def component_artifacts(component):
    return [MODEL_DIR / f"{component}.pt", MODEL_DIR / f"{component}.pt.ckpt",
            SCALER_DIR / f"{component}.joblib"]

def component_is_complete(component):
    entry = config_state["completed_components"].get(component)
    if not entry or entry.get("config_hash") != CONFIG_HASH:
        return False
    paths = component_artifacts(component)
    return all(path.is_file() and sha_file(path) == entry["sha256"].get(path.name)
               for path in paths)

atomic_json(CONFIG_PATH, config_state)
print("Drive folders: models/, scalers/, checkpoints/, forecasts/; Darts trains locally:", LOCAL_TRAIN_DIR)


## 9. Load trained models

In [ ]:
import joblib
from pytorch_lightning.callbacks import Callback, ModelCheckpoint
from types import MethodType

class DriveCheckpointMirror(Callback):
    """Mirror Darts' rolling best/last checkpoints after each completed training epoch."""
    def __init__(self, source_dir, drive_dir, base_model):
        self.source_dir, self.drive_dir, self.base_model = Path(source_dir), Path(drive_dir), Path(base_model)
    def sync(self):
        if self.base_model.is_file():
            atomic_copy(self.base_model, self.drive_dir / "_model.pth.tar")
        for pattern in ("best-*.ckpt", "last-*.ckpt"):
            for source in self.source_dir.glob(pattern):
                atomic_copy(source, self.drive_dir / source.name)
    def on_train_epoch_end(self, trainer, pl_module):
        self.sync()

training_model = ForecastModel(HORIZON, context_length=CONTEXT, tuned="default",
    split_date=SPLIT_AFTER, data_range=[TRAIN_START, VAL_END], past_covariates=[], future_covariates=[])
training_model.device = ACCELERATOR
training_df = training_model.process_data()

for number, component in enumerate(COMPONENTS, 1):
    print(f"[{number}/12] {component}")
    if component_is_complete(component):
        print(f"[{number}/12] {component} — compatible model found; training skipped")
        continue
    if not TRAIN_MISSING:
        raise RuntimeError(f"{component} is not trained. Set TRAIN_MISSING=True to train it.")
    final_paths = component_artifacts(component)
    present = [path.exists() for path in final_paths]
    if all(present):
        raise RuntimeError(f"Unregistered model files for {component} already exist; preserved at {MODEL_DIR}. Inspect before continuing.")
    if any(present):
        print(f"[{number}/12] incomplete transfer detected; new verified artifacts will replace only these unregistered files")

    wrapper = training_model.prepare_model(training_df.copy(), component)
    wrapper.work_dir = str(DARTS_WORK_DIR)
    wrapper.force_reset = False
    model_name = wrapper.get_model_name()
    local_model_dir = DARTS_WORK_DIR / model_name
    local_checkpoint_dir = local_model_dir / "checkpoints"
    drive_checkpoint_dir = CHECKPOINT_DIR / component
    drive_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_config = drive_checkpoint_dir / "compatibility.json"
    if checkpoint_config.exists():
        if checkpoint_config.read_text().strip() != CONFIG_HASH:
            raise RuntimeError(f"Incompatible checkpoint for {component}; preserved at {drive_checkpoint_dir}.")
    elif any(drive_checkpoint_dir.glob("*.ckpt")) or (drive_checkpoint_dir / "_model.pth.tar").exists():
        raise RuntimeError(f"Checkpoint for {component} has no compatibility record; preserved at {drive_checkpoint_dir}.")
    else:
        atomic_json(checkpoint_config, CONFIG_HASH)

    resume = any(drive_checkpoint_dir.glob("last-*.ckpt"))
    if resume:
        local_model_dir.mkdir(parents=True, exist_ok=True); local_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        base = drive_checkpoint_dir / "_model.pth.tar"
        if not base.is_file():
            raise RuntimeError(f"Checkpoint for {component} lacks _model.pth.tar; preserved and cannot be resumed safely.")
        atomic_copy(base, local_model_dir / base.name)
        for checkpoint in drive_checkpoint_dir.glob("*.ckpt"):
            atomic_copy(checkpoint, local_checkpoint_dir / checkpoint.name)
        print(f"[{number}/12] checkpoint found; resuming training")
    else:
        print(f"[{number}/12] Training started")

    wrapper.extra_callbacks = [DriveCheckpointMirror(
        local_checkpoint_dir, drive_checkpoint_dir, local_model_dir / "_model.pth.tar")]
    aligned = training_df.tail(HORIZON * ((len(training_df) - CONTEXT) // HORIZON) + CONTEXT)
    train_dates = aligned.loc[aligned["date"] <= pd.Timestamp(SPLIT_AFTER), "date"]
    val_dates = aligned.loc[aligned["date"] > pd.Timestamp(SPLIT_AFTER), "date"]
    if train_dates.max() != pd.Timestamp(TRAIN_END) or val_dates.min() != pd.Timestamp(VAL_START):
        raise RuntimeError("Unexpected train/validation boundary.")
    print("Train:", train_dates.min().date(), "-", train_dates.max().date(),
          "| Validation:", val_dates.min().date(), "-", val_dates.max().date())

    local_model_path = LOCAL_TRAIN_DIR / f"{component}.pt"
    (LOCAL_TRAIN_DIR / "save").mkdir(parents=True, exist_ok=True)
    with working_directory(LOCAL_TRAIN_DIR):
        model, scaler = wrapper.train(save_path=str(local_model_path), resume=resume)
    if not local_model_path.is_file() or not Path(str(local_model_path) + ".ckpt").is_file():
        raise RuntimeError(f"{component}: Darts did not create both model artifacts; checkpoints remain in {drive_checkpoint_dir}.")
    callback = next((cb for cb in model.trainer.callbacks
        if isinstance(cb, ModelCheckpoint) and cb.monitor == "val_loss"), None)
    if callback is None or callback.best_model_score is None:
        raise RuntimeError(f"{component}: best val_loss is unavailable; checkpoint files were preserved.")
    best_loss = float(callback.best_model_score.detach().cpu())
    best_epoch = int(callback.best_model_path.rsplit("epoch=", 1)[-1].split("-", 1)[0]) if "epoch=" in callback.best_model_path else -1
    print(f"Best epoch: {best_epoch}; validation loss: {best_loss:.8g}")

    # Verify local artifacts load before copying; Darts fit(load_best=True) restored minimum val_loss.
    loaded = TiDEModel.load(str(local_model_path), map_location=DEVICE,
        pl_trainer_kwargs={"accelerator": ACCELERATOR, "devices": 1})
    pending = [LOCAL_TRAIN_DIR / f"{component}.pending.pt",
               LOCAL_TRAIN_DIR / f"{component}.pending.pt.ckpt",
               LOCAL_TRAIN_DIR / f"{component}.pending.joblib"]
    for src, tmp in zip([local_model_path, Path(str(local_model_path)+".ckpt")], pending[:2]):
        atomic_copy(src, tmp)
    joblib.dump(scaler, pending[2])
    for tmp in pending:
        if not tmp.is_file() or tmp.stat().st_size == 0:
            raise RuntimeError(f"{component}: empty temporary artifact; checkpoints preserved.")
    for tmp, dest in zip(pending, final_paths):
        atomic_copy(tmp, dest)
    # Verify Drive copies before recording completion.
    TiDEModel.load(str(final_paths[0]), map_location=DEVICE,
        pl_trainer_kwargs={"accelerator": ACCELERATOR, "devices": 1})
    joblib.load(final_paths[2])
    checksums = {path.name: sha_file(path) for path in final_paths}
    config_state["completed_components"][component] = {
        "config_hash": CONFIG_HASH, "sha256": checksums,
        "best_epoch": best_epoch, "best_val_loss": best_loss,
    }
    atomic_json(CONFIG_PATH, config_state)
    # A finished model no longer needs the temporary resume checkpoints.
    shutil.rmtree(drive_checkpoint_dir)
    for tmp in pending:
        tmp.unlink(missing_ok=True)
    print(f"[{number}/12] Model saved; scaler saved; component complete")
    del model, loaded, wrapper
    import gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print("All 12 components are complete.")


## 10. Forecast next 7 days

In [ ]:
import joblib
from types import MethodType

MODEL_PATHS = {}
for component in COMPONENTS:
    if not component_is_complete(component):
        raise RuntimeError(f"No compatible completed model/scaler for {component}. Run training with TRAIN_MISSING=True.")
    MODEL_PATHS[component] = MODEL_DIR / f"{component}.pt"
    scaler_source = SCALER_DIR / f"{component}.joblib"
    scaler_local = LOCAL_TRAIN_DIR / "save" / f"scaler_{component}.save"
    scaler_local.parent.mkdir(parents=True, exist_ok=True)
    atomic_copy(scaler_source, scaler_local)
local_hyperparams = LOCAL_TRAIN_DIR / "hyperparams" / "default.env"
local_hyperparams.parent.mkdir(parents=True, exist_ok=True)
atomic_copy(REPO / "hyperparams/default.env", local_hyperparams)
print("Configuration and checksums verified for all 12 model/scaler pairs.")

def load_saved(component, wrapper):
    return TiDEModel.load(str(MODEL_PATHS[component]), map_location=DEVICE,
        pl_trainer_kwargs={"accelerator": ACCELERATOR, "devices": 1})

forecast_model = ForecastModel(HORIZON, context_length=CONTEXT, tuned="default",
    data_range=[TRAIN_START, LAST_DATE.strftime("%Y-%m-%d")],
    past_covariates=[], future_covariates=[])
forecast_model.device = ACCELERATOR
original_prepare_model = forecast_model.prepare_model

def prepare_saved_model(self, df, imf, mode="forecast"):
    wrapper = original_prepare_model(df, imf, mode=mode)
    wrapper.load_model = MethodType(lambda instance: load_saved(imf, instance), wrapper)
    return wrapper

forecast_model.prepare_model = MethodType(prepare_saved_model, forecast_model)
# Call the original 12-component prediction path; no training occurs in this section.
with working_directory(LOCAL_TRAIN_DIR):
    forecast = forecast_model.make_prediction()
forecast_values = np.asarray(forecast.univariate_values(), dtype=float)
expected_dates = pd.date_range(LAST_DATE + pd.Timedelta(days=1), periods=HORIZON, freq="D")
if not forecast.time_index.equals(expected_dates) or len(forecast_values) != HORIZON or not np.isfinite(forecast_values).all():
    raise RuntimeError("Forecast dates or values are invalid.")
forecast_table = pd.DataFrame({
    "date": expected_dates,
    "forecast_f10_7_sfu": forecast_values,
})
print("Последнее наблюдение:", LAST_DATE.date())
print("Прогноз:", expected_dates[0].date(), "-", expected_dates[-1].date())
if LAST_DATE.date() < datetime.now(timezone.utc).date():
    print("Прогноз построен после последней даты данных, не от текущей даты.")


## 11. Display and save results

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
display(forecast_table.style.format({"date": "{:%Y-%m-%d}", "forecast_f10_7_sfu": "{:.2f}"}))
recent = observed.tail(90)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent["date"], recent["real"], label="Наблюдения F10.7")
ax.plot(forecast_table["date"], forecast_table["forecast_f10_7_sfu"],
        "o-", label="Прогноз VMD-TiDE-12 (7 дней)")
ax.axvline(expected_dates[0], color="gray", linestyle="--", label="Начало прогноза")
ax.set(xlabel="Дата", ylabel="F10.7, sfu",
       title=f"F10.7: прогноз после {LAST_DATE:%Y-%m-%d}")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()
output_dir = RESULTS_DIR / "forecasts" / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex)
output_dir.mkdir(parents=True, exist_ok=False)
forecast_table.to_csv(output_dir / "forecast_7day.csv", index=False, date_format="%Y-%m-%d")
fig.savefig(output_dir / "forecast_7day.png", dpi=160, bbox_inches="tight")
write_json_new(output_dir / "provenance.json", {
    "last_observation": LAST_DATE.isoformat(),
    "training_configuration": signature(TRAIN_CONFIG),
    "source_commit": SOURCE_COMMIT,
    "components": {k: str(v) for k, v in MODEL_PATHS.items()},
})
plt.show()
print("Сохранено:", output_dir / "forecast_7day.csv", "\n", output_dir / "forecast_7day.png")

In [ ]:
test_start = pd.Timestamp(TEST_START).normalize()
test_end = min(pd.Timestamp(TEST_END).normalize(), LAST_DATE.normalize()) if TEST_END else LAST_DATE.normalize()
if test_start <= pd.Timestamp(VAL_END) or test_end < test_start:
    raise ValueError("The test interval must start after validation and contain available observations.")
history_start = max(pd.Timestamp(TRAIN_START), test_start - pd.Timedelta(days=CONTEXT + HORIZON + 14))
backtest_model = ForecastModel(HORIZON, context_length=CONTEXT, tuned="default",
    data_range=[history_start.strftime("%Y-%m-%d"), test_end.strftime("%Y-%m-%d")],
    past_covariates=[], future_covariates=[])
backtest_model.device = ACCELERATOR
backtest_prepare = backtest_model.prepare_model

def prepare_backtest_model(self, df, imf, mode="evaluate"):
    wrapper = backtest_prepare(df, imf, mode=mode)
    wrapper.load_model = MethodType(lambda instance: load_saved(imf, instance), wrapper)
    print(f"Backtest [{COMPONENTS.index(imf)+1}/12]: {imf}")
    return wrapper

backtest_model.prepare_model = MethodType(prepare_backtest_model, backtest_model)
print("Retrospective evaluation can take time; models are not retrained.")
print("Caution: full-history VMD uses future observations; MAE is not leakage-free.")
with working_directory(LOCAL_TRAIN_DIR):
    backtest_forecast, _ = backtest_model.forecast(mode="evaluate")
predicted = pd.DataFrame({"date": pd.to_datetime(backtest_forecast.time_index),
                          "forecast_f10_7_sfu": backtest_forecast.univariate_values()})
actual = observed.loc[(observed["date"] >= test_start) & (observed["date"] <= test_end), ["date", "real"]].copy()
actual = actual.rename(columns={"real": "observed_f10_7_sfu"})
backtest = actual.merge(predicted, on="date", how="left", validate="one_to_one").sort_values("date")
if backtest.empty or backtest["forecast_f10_7_sfu"].isna().any() or not np.isfinite(backtest["forecast_f10_7_sfu"]).all():
    raise RuntimeError("Backtest predictions do not cover the complete test interval.")
backtest["absolute_error_sfu"] = (backtest["forecast_f10_7_sfu"] - backtest["observed_f10_7_sfu"]).abs()
yearly_mae = backtest.groupby(backtest["date"].dt.year)["absolute_error_sfu"].mean().rename_axis("year").reset_index(name="mae_sfu")
test_mae = float(backtest["absolute_error_sfu"].mean())

fig_test, ax_test = plt.subplots(figsize=(14, 5))
ax_test.plot(backtest["date"], backtest["observed_f10_7_sfu"], label="Observed F10.7", color="royalblue", linewidth=1)
ax_test.plot(backtest["date"], backtest["forecast_f10_7_sfu"], label="VMD-TiDE: 7-day forecast", color="forestgreen", linewidth=1)
ax_test.set(xlabel="Date", ylabel="F10.7, sfu", title="Test interval: observations and 7-day forecast (full-history VMD)")
ax_test.legend(); ax_test.grid(alpha=0.3); fig_test.autofmt_xdate(); fig_test.tight_layout()
fig_test.savefig(output_dir / "test_observed_vs_forecast.png", dpi=160, bbox_inches="tight")
plt.show()

fig_year, ax_year = plt.subplots(figsize=(12, 4))
ax_year.plot(yearly_mae["year"], yearly_mae["mae_sfu"], marker="o", label="VMD-TiDE-12")
ax_year.set(xlabel="Year", ylabel="MAE, sfu", title="Mean absolute error by year (full-history VMD)")
ax_year.legend(); ax_year.grid(alpha=0.3); fig_year.tight_layout()
fig_year.savefig(output_dir / "test_mae_by_year.png", dpi=160, bbox_inches="tight")
plt.show()

fig_mae, ax_mae = plt.subplots(figsize=(6, 4))
bars = ax_mae.bar(["VMD-TiDE-12"], [test_mae], color="steelblue")
ax_mae.bar_label(bars, labels=[f"{test_mae:.3f}"], padding=3)
ax_mae.set(ylabel="MAE, sfu", title="Test mean absolute error (full-history VMD)")
ax_mae.set_ylim(0, max(test_mae * 1.2, 0.01)); ax_mae.grid(axis="y", alpha=0.3)
fig_mae.tight_layout(); fig_mae.savefig(output_dir / "test_mae.png", dpi=160, bbox_inches="tight")
plt.show()
backtest.to_csv(output_dir / "test_predictions.csv", index=False, date_format="%Y-%m-%d")
yearly_mae.to_csv(output_dir / "test_mae_by_year.csv", index=False)
print(f"Test interval: {test_start.date()} - {test_end.date()}; MAE: {test_mae:.3f} sfu; days: {len(backtest)}")
print("Saved test figures and data:", output_dir)